# 06 · NEWS2 vs the first model: split, train, calibrate, choose, register

**Who this is for:** you, after notebook 05 (which saved the training table).
**You should already know:** the patient-hour table, labels and NEWS2 (notebook 05).

**By the end you will be able to:**
- split data **by time** and explain why a random split cheats
- train two candidates (logistic regression and LightGBM) and handle a 1%-positive problem without breaking the model
- **calibrate** probabilities so "30%" means 30%
- turn a score into **alerts** with a budget nurses can live with
- choose a **champion** on validation with pass marks against NEWS2, then judge it once on the test set
- log runs to **MLflow**, register the champion and load it by its **`@champion` alias**

**Outline**
1. Setup
2. Split by time
3. Why not a random split?
4. The baseline: NEWS2
5. Candidate 1: logistic regression
6. Candidate 2: LightGBM, and the class-weight trap
7. Calibration
8. Choosing the champion on validation
9. The final exam: test set
10. MLflow: track, register, alias
11. Exercise
12. Pitfalls and next steps

Code used: `heavden/ml/train.py`, `heavden/ml/evaluate.py`, `heavden/ml/tracking.py` (tests in `tests/test_train.py`).

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from heavden.ml import evaluate, tracking, train


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
path = ROOT / "data" / "gold_local" / "patient_hours_baseline.parquet"
if not path.exists():
    raise FileNotFoundError("Run notebook 05 first: it saves the training table.")
table = pd.read_parquet(path)
known, positives = table["label"].notna().sum(), int(table["label"].sum())
print(f"{len(table):,} patient-hours | known labels: {known:,} | positives: {positives}")

## 2. Split by time

The model will always predict the **future** from the **past**, so it must be tested the same way:

```
days 1 ────────── 9 │gap│ 10 ── 11 │gap│ 12 ──── 14
       TRAIN        6 h    VALID     6 h    TEST
   learn patterns      choose & calibrate   final exam (used once)
```

The 6-hour gaps stop a training row's look-ahead window from overlapping validation rows.

In [ ]:
tr, va, te = train.time_split(table)
pd.DataFrame(
    {
        name: {
            "from": d["prediction_ts"].min(),
            "to": d["prediction_ts"].max(),
            "patient-hours": len(d),
            "positives": int(d["label"].sum()),
            "positive rate": f"{d['label'].mean():.2%}",
        }
        for name, d in {"train": tr, "valid": va, "test": te}.items()
    }
)

Only ~100–160 positives in validation and test. Every metric below has real uncertainty, and that's normal for rare events.

## 3. Why not a random split?

A random split puts **neighbouring hours of the same patient** in both training and test. The model then recognises the patient's last few hours instead of learning general warning signs.

In [ ]:
columns = train.model_columns(table)
labelled = table[table["label"].notna()]
shuffled = labelled.sample(frac=1, random_state=0)
cut = int(0.7 * len(shuffled))
r_train, r_test = shuffled.iloc[:cut], shuffled.iloc[cut:]

time_model = train.fit_logistic(tr, columns)
random_model = train.fit_logistic(r_train, columns)
comparison = {
    "random split (looks great)": evaluate.ranking_metrics(
        r_test["label"], random_model.predict_proba(train.to_matrix(r_test, columns))[:, 1]
    ),
    "time split (honest)": evaluate.ranking_metrics(
        te["label"], time_model.predict_proba(train.to_matrix(te, columns))[:, 1]
    ),
}
pd.DataFrame(comparison).T[["auroc", "auprc"]].round(3)

If the random split looks better, that's the leak showing. Even when the difference is small, the time split is the only one that matches how the model will be used.

## 4. The baseline: NEWS2

NEWS2 is already a risk score: higher means sicker. To compare fairly, we rank patient-hours by NEWS2 (ties broken by lowest SpO2) and give it the **same alert budget** as the model.

### Alerts, not hours
A score becomes useful when it raises **alerts**. Our rule (the same one the dashboards use):
- a patient is **High** when their score is at or above a threshold;
- an **alert** fires when a patient **enters** High; staying High doesn't re-alert;
- the same patient doesn't alert again for **6 hours** (to limit alarm fatigue).

Nurses can act on only so many alerts. We assume a budget of **2 alerts per nurse per 12-hour shift**, with **1 nurse per 5 patients**. The threshold is set on validation as the lowest one that keeps alerts within that budget.

In [ ]:
budget = evaluate.AlertBudget()
b = budget
per_shift, patients, hours = b.alerts_per_nurse_per_shift, b.patients_per_nurse, b.shift_hours
print(f"{per_shift:g} alerts / ({patients:g} patients x {hours:g} h)")
print(f"= {b.alerts_per_patient_hour:.4f} alerts per patient-hour")
print(f"= about {b.alerts_per_patient_hour * len(va):.0f} alerts allowed over the validation block")

In [ ]:
y_va, y_te = va["label"].to_numpy(int), te["label"].to_numpy(int)
news2_va, news2_te = train.news2_scores(va), train.news2_scores(te)
news2_threshold = evaluate.threshold_for_alert_budget(va, news2_va, budget)  # on validation
baseline_valid = evaluate.evaluate(va, news2_va, news2_threshold, probabilities=False)
pd.Series(baseline_valid).round(3).to_frame("NEWS2 on validation")

**Reading these** (two levels, answering different questions):
- **Per patient-hour (ranking):** *AUROC* and *AUPRC*. AUPRC is the headline for rare events (a random guess scores the positive rate, about 0.01).
- **Per alert (what a nurse experiences):** *alerts_per_nurse_shift* is the alert load; *alert_precision* = share of alerts followed by an escalation within 6 hours; *escalations_flagged* = share of escalations where the patient was High at some point in the 6 hours before; *median_warning_hours* = how early.
- *high_hour_share* is the share of hours spent in High. It is **not** the alert count: a patient who stays High for 10 hours raises one alert.

## 5. Candidate 1: logistic regression

A weighted sum of the features passed through a sigmoid: simple, fast and explainable. Missing values are filled with the training median, features are standardised, and `C=0.1` adds regularisation (protection against over-fitting ~400 positives).

In [ ]:
lr = train.fit_logistic(tr, columns)
coef = pd.Series(lr[-1].coef_[0], index=columns)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(12).round(3).to_frame(
    "weight (standardised)"
)

Positive weights push risk up, negative ones down. **Be careful reading them:** many features are strongly correlated (e.g. `resp_rate_mean_1h`, `_3h`, `_6h` and `news2_resp_rate` all describe breathing), so the model can split credit between them in odd ways. A *negative* weight on the 6-hour breathing average next to positive weights on recent breathing effectively means *"breathing faster than this patient's own recent baseline"*, which is a trend. Individual weights in a model with correlated inputs are not a reliable explanation. Notebook 07 uses SHAP for that.

## 6. Candidate 2: LightGBM, and the class-weight trap

LightGBM builds many small decision trees that can capture non-linear patterns. It stops adding trees when **validation average precision** stops improving.

With only ~0.7% positives it's tempting to weight positives ~150×. In Phase 0 that made the model *worse than NEWS2*. Let's see why:

In [ ]:
heavy = train.fit_lightgbm(tr, va, columns, pos_weight=(tr["label"] == 0).sum() / tr["label"].sum())
light = train.fit_lightgbm(tr, va, columns, pos_weight=1.0)
pd.DataFrame(
    {
        f"pos_weight={w:.0f}": evaluate.ranking_metrics(
            y_va, m.predict_proba(train.to_matrix(va, columns))[:, 1]
        )
        | {"trees": m.best_iteration_}
        for w, m in {heavy.get_params()["scale_pos_weight"]: heavy, 1.0: light}.items()
    }
).T[["auroc", "auprc", "trees"]].round(3)

**Interpretation:** the heavy weight makes each positive count like ~150 negatives, so the model overfits those few hundred cases and early-stops after a handful of trees. A light touch (weight 1, i.e. none) generalises better. Ranking metrics don't need balanced classes, and calibration fixes the probabilities afterwards.

In [ ]:
importance = pd.Series(light.booster_.feature_importance("gain"), index=columns)
importance.sort_values(ascending=False).head(12).round(0).to_frame("gain")

## 7. Calibration

A risk score is only useful at the bedside if it's **honest**: of patients given 30%, about 30% should escalate. **Isotonic calibration** learns a monotone map from raw scores to observed rates on the validation set. It changes the numbers but not the ranking.

In [ ]:
results, baseline = train.train_candidates(tr, va, budget)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, r in zip(axes, results, strict=True):
    raw = r.model.raw_scores(te)
    cal = r.model.predict_proba(te)[:, 1]
    for name, p in {"raw": raw, "calibrated": cal}.items():
        t = evaluate.calibration_table(y_te, p, n_bins=8)
        ax.plot(t["predicted"], t["observed"], "o-", label=name)
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, lim], [0, lim], ls="--", color="grey", lw=1)
    ax.set_title(f"{r.name} (test set)")
    ax.set_xlabel("predicted risk")
    ax.set_ylabel("observed rate")
    ax.legend()
plt.tight_layout()

Points on the dashed line are perfectly calibrated. With so few positives the high-risk bins are noisy. That's why the calibration check in the retraining gates looks at the **Brier score** across all predictions, not at a single bin.

One more detail in `train.calibrate`: the top step of an isotonic fit can rest on a handful of hours that all escalated, which would make the model say **100%**. Two anchor points keep both ends honest, so the model never claims certainty.

## 8. Choosing the champion on validation

Pass marks (Plan.md §8 and §10): a candidate must beat NEWS2 on **validation AUPRC** and **flag at least as many escalations** when both raise alerts within the same budget. Among those that pass, the best validation AUPRC wins, whether simple or not.

In [ ]:
board = pd.DataFrame({"NEWS2": baseline} | {r.name: r.valid_metrics for r in results}).T
board["passes gates"] = [None] + [train.passes_gates(r.valid_metrics, baseline) for r in results]
champion = train.select_champion(results, baseline)
print("champion:", champion.name)
cols = ["auroc", "auprc", "alerts_per_nurse_shift", "alert_precision", "escalations_flagged"]
board[[*cols, "high_hour_share", "passes gates"]].round(3)

## 9. The final exam: test set

The test set is used **once**, for the champion only. The alert thresholds are the ones fixed on validation, as they would be in production, so the alert load on test can drift a little from the budget.

In [ ]:
p_va = champion.model.predict_proba(va)[:, 1]
p_te = champion.model.predict_proba(te)[:, 1]
threshold = champion.valid_metrics["threshold"]  # fixed on validation

final = pd.DataFrame(
    {
        "NEWS2": evaluate.evaluate(te, news2_te, news2_threshold, probabilities=False),
        f"champion ({champion.name})": evaluate.evaluate(te, p_te, threshold),
    }
).T
cols = ["auroc", "auprc", "alerts_per_nurse_shift", "alert_precision", "escalations"]
final[[*cols, "escalations_flagged", "median_warning_hours", "high_hour_share"]].round(3)

**Interpretation:**
- Per hour, the champion ranks much better: AUPRC is roughly double NEWS2's.
- Per alert, at about the same alert load, the champion **flags more of the escalations in advance**, with a similar share of alerts followed by an escalation (about 1 in 30). For both scores most alerts are not followed by an escalation within 6 hours: that's normal for early-warning systems, and why the budget matters.
- NEWS2 scores are whole numbers, so its threshold can't hit the budget exactly; it lands a little below 2 alerts per nurse per shift.
- With fewer than 30 escalations in test, notebook 07 puts intervals on these numbers.

Keep the honest caveat from Plan.md §8 in mind: our simulated deteriorations are smooth, so real-world gains would likely be smaller.

### Why count alerts, not hours? A lesson learned in this project
An earlier version set the threshold so that **3.3% of patient-hours** were High, assuming that equalled "2 alerts per nurse per shift". But a deteriorating patient's NEWS2 stays High for hours (one alert), while the model's risk can *flicker* around the threshold (a new alert at every re-entry). Compare both scores at the **same share of High hours**:

In [ ]:
same_hours = {}
for name, scores in {"NEWS2": news2_te, "champion": p_te}.items():
    cut = evaluate.threshold_for_fraction(scores, 0.033)  # top 3.3% of hours
    m = evaluate.alert_metrics(te, scores, cut)
    same_hours[name] = {k: m[k] for k in ("high_hour_share", "alerts", "escalations_flagged")}
pd.DataFrame(same_hours).T.round(3)

Same share of High hours, very different alert counts. Counting **alerts** is what matches a nurse's workload, so thresholds are now set from the alert budget directly.

## 10. MLflow: track, register, alias

MLflow records each candidate as a **run** (parameters, metrics, the calibrated model). The champion is **registered** as a numbered version and given the alias **`@champion`**. Anything that scores patients loads `models:/deterioration_risk@champion`, so promoting a new model is just moving the alias.

Locally MLflow uses a SQLite store in `data/mlflow/` (git-ignored). In Databricks the same calls go to Unity Catalog. (A warning about an unresolved `pip` version is harmless: this project installs packages with `uv`.)

In [ ]:
tracking.use_local_store(ROOT / "data" / "mlflow")
logged = {
    r.name: tracking.log_candidate(r, te, tags={"is_champion": str(r is champion)}) for r in results
}
version = tracking.register_champion(logged[champion.name])
print(f"registered '{tracking.LOCAL_MODEL_NAME}' version {version} as @champion")

In [ ]:
loaded = mlflow.sklearn.load_model(f"models:/{tracking.LOCAL_MODEL_NAME}@champion")
same = np.allclose(loaded.predict_proba(te)[:, 1], p_te)
print("loaded by alias; predictions identical:", same)

To browse runs in the MLflow UI, run this in a terminal from the repo root, then open http://127.0.0.1:5000:

```bash
uv run mlflow ui --backend-store-uri sqlite:///data/mlflow/mlflow.db
```

Each time you re-run this notebook a new version is registered and `@champion` moves to it. That's the same mechanism the retraining workflow uses, with a human approval in between.

## 11. Exercise: a stricter alert budget

The ward manager says nurses can only handle **1 alert per nurse per shift**.
1. create `evaluate.AlertBudget(alerts_per_nurse_per_shift=1)`
2. recompute the thresholds on validation (for NEWS2 and the champion) and the test metrics
3. how much recall do you lose? Does the champion's advantage over NEWS2 grow or shrink?

In [ ]:
# Try it yourself
strict = None

strict_results = None

In [ ]:
# Answer (run after trying)
strict = evaluate.AlertBudget(alerts_per_nurse_per_shift=1)
strict_thresholds = {
    "NEWS2": (news2_te, evaluate.threshold_for_alert_budget(va, news2_va, strict)),
    "champion": (p_te, evaluate.threshold_for_alert_budget(va, p_va, strict)),
}
strict_results = pd.DataFrame(
    {name: evaluate.alert_metrics(te, s, t, strict) for name, (s, t) in strict_thresholds.items()}
).T[["alerts_per_nurse_shift", "alert_precision", "escalations_flagged", "median_warning_hours"]]
strict_results.round(3)

A tighter budget raises precision (fewer, more certain alerts) but flags fewer of the escalations, and later. Choosing the budget is a **clinical** decision, and the platform should make the trade-off visible rather than hide it.

## 12. Pitfalls and next steps

**Pitfall: peeking at the test set.** Choosing models or thresholds on test makes the final number optimistic. Here everything is chosen on validation, and test is scored once.

**Pitfall: extreme class weights.** They don't help ranking and can wreck it. Use none or modest weights, and calibrate afterwards.

**Pitfall: thresholds that ignore capacity.** "Alert if risk > 50%" means nothing to a ward. Thresholds should come from an alert budget.

**Pitfall: reading too much into small numbers.** About 160 test positives means differences of a few points of AUPRC can be noise. Notebook 07 adds confidence intervals.

**Optional extension:** add a third candidate (e.g. LightGBM with `num_leaves=7`) inside `heavden/ml/train.py` and see whether the champion changes. Add a test for it too.

**Next notebook:** `07_evaluation_deep_dive`. Confidence intervals, slice fairness (site, age, sex), warning time before escalation, and SHAP explanations for individual patients.